<a href="https://colab.research.google.com/github/jimenz2/territorial/blob/main/Umodelomontled88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================
# Script de preparación de base de datos
# para análisis estocástico de Plan México 2025-2030
# ========================================

# 1️⃣ Instalación de bibliotecas necesarias
!pip install PyPDF2 python-docx docx2txt openpyxl pandas numpy matplotlib seaborn nltk scikit-learn

# 2️⃣ Importar librerías
import os
import pandas as pd
import numpy as np
from PyPDF2 import PdfReader
import docx2txt
import nltk
from sklearn.preprocessing import MinMaxScaler

# Descargar recursos de NLTK para procesamiento de texto
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 3️⃣ Funciones para extraer texto
def extraer_pdf(path):
    texto = ""
    reader = PdfReader(path)
    for page in reader.pages:
        texto += page.extract_text() + " "
    return texto

def extraer_docx(path):
    return docx2txt.process(path)

def extraer_excel(path):
    return pd.read_excel(path)

# 4️⃣ Carpeta donde subes tus archivos en Colab
# Los archivos seleccionados por el usuario están en /content/
carpeta = "/content/"
archivos = [
    "2026_03_12_cgpe_2026.pdf",
    "2026_03_12_68b4162a4d926-1IG-INFORME-INTEGRADO-FINAL_26_08_2025.pdf",
    "2026_03_12_6_Informe_de_Labores_de_la_Secretar_a_de_la_Funci_n_P_blica.pdf",
    "2026_03_12_PND_2025-2030_v250226_14.pdf"
]

# 5️⃣ Inicializamos listas para almacenar los datos
sectores, programas, indicadores, metas_cuantitativas, metas_cualitativas, inversiones, plazos, fuentes = [], [], [], [], [], [], [], []

# 6️⃣ Función básica de extracción de información
# Nota: esta función es base, luego puedes mejorar con NLP avanzado
def procesar_texto(texto, fuente):
    lineas = texto.split("\n")
    for linea in lineas:
        linea = linea.strip()
        if len(linea) > 10:  # Filtrar líneas muy cortas
            # Ejemplo: puedes separar con ":" para identificar indicador y meta
            if ":" in linea:
                parts = linea.split(":")
                sector = parts[0].strip()
                info = parts[1].strip()
                sectores.append(sector)
                programas.append(info)
                indicadores.append(np.nan)
                metas_cuantitativas.append(np.nan)
                metas_cualitativas.append(np.nan)
                inversiones.append(np.nan)
                plazos.append(np.nan)
                fuentes.append(fuente)

# 7️⃣ Procesar cada archivo según su tipo
for archivo in archivos:
    path = os.path.join(carpeta, archivo)
    if archivo.endswith(".pdf"):
        texto = extraer_pdf(path)
        procesar_texto(texto, archivo)
    elif archivo.endswith(".docx"):
        texto = extraer_docx(path)
        procesar_texto(texto, archivo)
    elif archivo.endswith(".xlsx") or archivo.endswith(".xls"):
        df_excel = extraer_excel(path)
        # Asume que el Excel tiene columnas "sector" y "programa"
        for idx, row in df_excel.iterrows():
            sectores.append(row.get("sector", np.nan))
            programas.append(row.get("programa", np.nan))
            indicadores.append(row.get("indicador", np.nan))
            metas_cuantitativas.append(row.get("meta_cuantitativa", np.nan))
            metas_cualitativas.append(row.get("meta_cualitativa", np.nan))
            inversiones.append(row.get("inversion_mxn", np.nan))
            plazos.append(row.get("plazo", np.nan))
            fuentes.append(archivo)

# 8️⃣ Crear DataFrame final
df_plan = pd.DataFrame({
    "sector": sectores,
    "programa": programas,
    "indicador": indicadores,
    "meta_cuantitativa": metas_cuantitativas,
    "meta_cualitativa": metas_cualitativas,
    "inversion_mxn": inversiones,
    "plazo": plazos,
    "fuente": fuentes
})

# 9️⃣ Normalización de columnas numéricas para análisis estocástico
scaler = MinMaxScaler()
for col in ["meta_cuantitativa", "inversion_mxn"]:
    if col in df_plan.columns:
        df_plan[col] = pd.to_numeric(df_plan[col], errors='coerce')
        df_plan[col] = scaler.fit_transform(df_plan[[col]].fillna(0))

# 10️⃣ Guardar DataFrame listo para análisis
df_plan.to_csv("/content/base_datos_plan_mx_2025_2030.csv", index=False)

print("✅ Base de datos generada y normalizada para análisis.")
df_plan.head(10)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_plan)

## SCRIPT BASE: ANÁLISIS ESTOCÁSTICO PLAN MX 2025-2030

In [ ]:
# 1️⃣ Instalar bibliotecas (si no están)
!pip install pandas numpy matplotlib plotly openpyxl --quiet

# 2️⃣ Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# 3️⃣ Cargar datos
# Usamos el archivo generado en el paso anterior
df = pd.read_csv('/content/base_datos_plan_mx_2025_2030.csv')

# 4️⃣ Revisión rápida
print("Primeras filas del dataset:")
display(df.head())

# 5️⃣ Normalización de datos
# Convertir porcentajes a escala 0-1
# Note: 'porcentaje' column is not in df_plan, 'meta_cuantitativa' and 'inversion_mxn' are already normalized.
# This section might need adjustment based on the actual columns and their state.
if 'porcentaje' in df.columns:
    df['porcentaje'] = df['porcentaje'] / 100

# Convertir inversión o montos a millones
if 'inversion_mxn' in df.columns:
    # df_plan['inversion_mxn'] is already normalized, so this might not be needed or needs to be adapted.
    df['inversion_mxn'] = df['inversion_mxn'] / 1_000_000

# Codificación de variables categóricas
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].astype('category').cat.codes

# 6️⃣ Simulación Monte Carlo de cumplimiento de metas
simulaciones = 10000
resultados = []

# Supongamos que las metas pueden variar ±10%
# Note: df_plan has 'meta_cuantitativa', not 'meta'. Adjusting this line to use 'meta_cuantitativa'.
# Also, 'meta_cuantitativa' is already normalized. Consider if the +/- 10% variation is still appropriate on a normalized scale.
for _ in range(simulaciones):
    # Using 'meta_cuantitativa' from df_plan. Ensure this column is suitable for direct simulation if it's already normalized.
    if 'meta_cuantitativa' in df.columns:
        scenario = df['meta_cuantitativa'] * np.random.normal(1, 0.1, size=len(df))
    else:
        # Fallback or error handling if 'meta_cuantitativa' is not found (though it should be from previous step)
        print("Warning: 'meta_cuantitativa' column not found for simulation.")
        scenario = np.zeros(len(df)) # Placeholder
    resultados.append(scenario.sum())

resultados = np.array(resultados)

# Estadísticas básicas
print("Valor esperado del portafolio de metas:", np.mean(resultados))
print("Desviación estándar:", np.std(resultados))
print("Percentiles 5%, 50%, 95%:", np.percentile(resultados, [5, 50, 95]))

# 7️⃣ Gráfica de distribución de escenarios
plt.figure(figsize=(10,6))
plt.hist(resultados, bins=50, color='skyblue', edgecolor='black')
plt.title('Distribuci\u00f3n de resultados Monte Carlo de metas PND 2025-2030')
plt.xlabel('Cumplimiento total de metas (normalizado)')
plt.ylabel('Frecuencia')
plt.show()

# 8️⃣ Gráfica interactiva con Plotly
fig = px.histogram(resultados, nbins=50, title='Distribuci\u00f3n de resultados Monte Carlo (Interactiva)')
fig.update_layout(xaxis_title='Cumplimiento total de metas', yaxis_title='Frecuencia')
fig.show()

# 9️⃣ Análisis de correlación entre indicadores
# Note: Correlation needs numerical columns. df_plan has 'meta_cuantitativa' and 'inversion_mxn' as normalized numerical columns.
# Other columns are text or NaN. Further feature engineering might be needed to get more numerical indicators for correlation.
numerical_df = df.select_dtypes(include=np.number)
# Drop columns that are entirely constant (like all zeros) or all NaN, as they yield NaN correlations.
numerical_df = numerical_df.loc[:, (numerical_df != numerical_df.iloc[0]).any() & numerical_df.notna().any()]

if not numerical_df.empty:
    correlation = numerical_df.corr()
    # Fill NaNs in the correlation matrix with 0 for plotting purposes
    correlation_for_plot = correlation.fillna(0)

    plt.figure(figsize=(10,8))
    plt.matshow(correlation_for_plot, fignum=1)
    plt.colorbar()
    plt.title('Matriz de correlaci\u00f3n entre indicadores')
    plt.show()
else:
    print("No sufficient numerical data available to compute a meaningful correlation matrix.")

print("Script base listo para an\u00e1lisis estoc\u00e1stico y sist\u00e9mico de pol\u00edticas p\u00fablicas.")

In [ ]:
# ===========================================
# SCRIPT AVANZADO: PORTAFOLIO DE POLÍTICAS
# Google Colab - Plan México 2025-2030
# ===========================================

!pip install pandas numpy matplotlib seaborn plotly openpyxl --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# 1️⃣ Cargar datos
# Archivo con columnas: 'sector', 'indicador', 'valor_base', 'meta', 'fecha_inicio', 'fecha_fin', 'tipo_variable', 'inversion_mxn'
df = pd.read_csv('plan_mexico_2025_2030.csv')
print("Primeras filas:")
display(df.head())

# 2️⃣ Normalización de datos
# Convertir valores monetarios a millones para homogeneidad
if 'inversion_mxn' in df.columns:
    df['inversion_mxn'] = df['inversion_mxn'] / 1_000_000

# 3️⃣ Calcular ponderaciones por sector (tipo portafolio)
df['peso_sector'] = df.groupby('sector')['inversion_mxn'].transform(lambda x: x / x.sum())

# 4️⃣ Simulación Monte Carlo estilo portafolio
simulaciones = 10000
sectores = df['sector'].unique()
resultados_portafolio = []

for _ in range(simulaciones):
    rendimiento_total = 0
    for sector in sectores:
        df_sector = df[df['sector']==sector]
        # Variación aleatoria ±10% por indicador
        escenario = df_sector['meta'] * np.random.normal(1, 0.10, size=len(df_sector))
        # Rendimiento ponderado por inversión
        rendimiento_total += np.sum(escenario * df_sector['peso_sector'])
    resultados_portafolio.append(rendimiento_total)

resultados_portafolio = np.array(resultados_portafolio)

# 5️⃣ Estadísticas del portafolio
mean_rend = resultados_portafolio.mean()
std_rend = resultados_portafolio.std()
perc_5, perc_95 = np.percentile(resultados_portafolio, [5,95])

print(f"Valor Esperado del Portafolio: {mean_rend:.2f}")
print(f"Riesgo (Desviación estándar): {std_rend:.2f}")
print(f"Escenario Pesimista (5%): {perc_5:.2f}")
print(f"Escenario Optimista (95%): {

In [ ]:
# ========================================
# Script de preparación de base de datos
# para análisis estocástico de Plan México 2025-2030
# ========================================

# 1️⃣ Instalación de bibliotecas necesarias
!pip install PyPDF2 python-docx openpyxl pandas numpy matplotlib seaborn nltk sklearn

# 2️⃣ Importar librerías
import os
import pandas as pd
import numpy as np
from PyPDF2 import PdfReader
import docx2txt
import nltk
from sklearn.preprocessing import MinMaxScaler

# Descargar recursos de NLTK para procesamiento de texto
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 3️⃣ Funciones para extraer texto
def extraer_pdf(path):
    texto = ""
    reader = PdfReader(path)
    for page in reader.pages:
        texto += page.extract_text() + " "
    return texto

def extraer_docx(path):
    return docx2txt.process(path)

def extraer_excel(path):
    return pd.read_excel(path)

# 4️⃣ Carpeta donde subes tus archivos en Colab
carpeta = "/content/plan_mx_docs/"
archivos = os.listdir(carpeta)

# 5️⃣ Inicializamos listas para almacenar los datos
sectores, programas, indicadores, metas_cuantitativas, metas_cualitativas, inversiones, plazos, fuentes = [], [], [], [], [], [], [], []

# 6️⃣ Función básica de extracción de información
# Nota: esta función es base, luego puedes mejorar con NLP avanzado
def procesar_texto(texto, fuente):
    lineas = texto.split("\n")
    for linea in lineas:
        linea = linea.strip()
        if len(linea) > 10:  # Filtrar líneas muy cortas
            # Ejemplo: puedes separar con ":" para identificar indicador y meta
            if ":" in linea:
                parts = linea.split(":")
                sector = parts[0].strip()
                info = parts[1].strip()
                sectores.append(sector)
                programas.append(info)
                indicadores.append(np.nan)
                metas_cuantitativas.append(np.nan)
                metas_cualitativas.append(np.nan)
                inversiones.append(np.nan)
                plazos.append(np.nan)
                fuentes.append(fuente)

# 7️⃣ Procesar cada archivo según su tipo
for archivo in archivos:
    path = os.path.join(carpeta, archivo)
    if archivo.endswith(".pdf"):
        texto = extraer_pdf(path)
        procesar_texto(texto, archivo)
    elif archivo.endswith(".docx"):
        texto = extraer_docx(path)
        procesar_texto(texto, archivo)
    elif archivo.endswith(".xlsx") or archivo.endswith(".xls"):
        df_excel = extraer_excel(path)
        # Asume que el Excel tiene columnas "sector" y "programa"
        for idx, row in df_excel.iterrows():
            sectores.append(row.get("sector", np.nan))
            programas.append(row.get("programa", np.nan))
            indicadores.append(row.get("indicador", np.nan))
            metas_cuantitativas.append(row.get("meta_cuantitativa", np.nan))
            metas_cualitativas.append(row.get("meta_cualitativa", np.nan))
            inversiones.append(row.get("inversion_mxn", np.nan))
            plazos.append(row.get("plazo", np.nan))
            fuentes.append(archivo)

# 8️⃣ Crear DataFrame final
df_plan = pd.DataFrame({
    "sector": sectores,
    "programa": programas,
    "indicador": indicadores,
    "meta_cuantitativa": metas_cuantitativas,
    "meta_cualitativa": metas_cualitativas,
    "inversion_mxn": inversiones,
    "plazo": plazos,
    "fuente": fuentes
})

# 9️⃣ Normalización de columnas numéricas para análisis estocástico
scaler = MinMaxScaler()
for col in ["meta_cuantitativa", "inversion_mxn"]:
    if col in df_plan.columns:
        df_plan[col] = pd.to_numeric(df_plan[col], errors='coerce')
        df_plan[col] = scaler.fit_transform(df_plan[[col]].fillna(0))

# 10️⃣ Guardar DataFrame listo para análisis
df_plan.to_csv("/content/base_datos_plan_mx_2025_2030.csv", index=False)

print("✅ Base de datos generada y normalizada para análisis.")
df_plan.head(10)

In [ ]:
# ===========================================
# OPTIMIZACIÓN DE PORTAFOLIO DE POLÍTICAS
# Google Colab - Plan México 2025-2030
# Basado en Teoría Moderna de Portafolio (Markowitz)
# ===========================================

!pip install pandas numpy matplotlib seaborn cvxpy plotly --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp

# 1️⃣ Cargar datos
df = pd.read_csv('plan_mexico_2025_2030.csv')
# Ejemplo de columnas: 'sector', 'indicador', 'meta', 'inversion_mxn'
df['inversion_mxn'] = df['inversion_mxn'] / 1_000_000  # convertir a millones

# 2️⃣ Agrupar por sector (rendimiento promedio ponderado)
sectores = df['sector'].unique()
rendimientos = {}
for sector in sectores:
    df_sec = df[df['sector']==sector]
    peso = df_sec['inversion_mxn'] / df_sec['inversion_mxn'].sum()
    rendimientos[sector] = np.sum(df_sec['meta'] * peso)

rendimientos = pd.Series(rendimientos)

# 3️⃣ Simulación Monte Carlo para volatilidad (riesgo)
simulaciones = 10000
mc_matrix = []
for _ in range(simulaciones):
    simulacion = []
    for sector in sectores:
        df_sec = df[df['sector']==sector]
        peso = df_sec['inversion_mxn'] / df_sec['inversion_mxn'].sum()
        # Simular ±10% de variación
        escenario = df_sec['meta'] * np.random.normal(1,0.10,len(df_sec))
        simulacion.append(np.sum(escenario * peso))
    mc_matrix.append(simulacion)

mc_matrix = np.array(mc_matrix)
cov_matrix = np.cov(mc_matrix.T)  # matriz de covarianza entre sectores

# 4️⃣ Optimización de portafolio
# Variables de decisión: ponderaciones por sector
w = cp.Variable(len(sectores))

# Función objetivo: minimizar riesgo (varianza)
retorno_esperado = rendimientos.values @ w
riesgo = cp.quad_form(w, cov_matrix)
objetivo = cp.Maximize(retorno_esperado - 0.5*riesgo)  # trade-off retorno/riesgo

# Restricciones: pesos positivos y suman 1
restricciones = [cp.sum(w) == 1, w >= 0]

prob = cp.Problem(objetivo, restricciones)
prob.solve()

# 5️⃣ Resultados óptimos
opt_weights = pd.Series(w.value, index=sectores).sort_values(ascending=False)
print("📊 Ponderaciones óptimas por sector:")
display(opt_weights)

# 6️⃣ Riesgo y retorno del portafolio óptimo
ret_opt = np.sum(rendimientos.values * w.value)
riesgo_opt = np.sqrt(w.value.T @ cov_matrix @ w.value)
print(f"Retorno esperado óptimo: {ret_opt:.2f}")
print(f"Riesgo (desviación estándar) óptimo: {riesgo_opt:.2f}")

# 7️⃣ Gráfica de distribución de portafolio simulado
plt.figure(figsize=(10,6))
sns.histplot(mc_matrix @ w.value, bins=50, kde=True, color='green')
plt.title("Distribución de cumplimiento del portafolio optimizado")
plt.xlabel("Cumplimiento ponderado de metas")
plt.ylabel("Frecuencia")
plt.grid(True)
plt.show()

# 8️⃣ Heatmap correlación entre sectores
plt.figure(figsize=(12,8))
sns.heatmap(pd.DataFrame(mc_matrix, columns=sectores).corr(), annot=True, cmap="coolwarm")
plt.title("Correlación entre sectores - Portafolio optimizado")
plt.show()

print("✅ Portafolio optimizado listo: ponderaciones, riesgo-retorno y correlaciones para análisis estratégico.")

# ===========================================

# SCRIPT AVANZADO: ANÁLISIS ESTOCASTICO DE POLÍTICAS
# Google Colab - Portafolio de metas sectoriales

# ===========================================

In [ ]:
!pip install pandas numpy matplotlib plotly openpyxl --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# 1️⃣ Cargar datos
# Usamos el archivo generado en el paso anterior
df = pd.read_csv('/content/base_datos_plan_mx_2025_2030.csv')
print("Primeras filas:")
display(df.head())

# 2️⃣ Normalización de datos
# Porcentaje 0-1, inversión en millones
# Note: 'porcentaje' column is not in df_plan. 'inversion_mxn' is already normalized.
if 'porcentaje' in df.columns:
    df['porcentaje'] = df['porcentaje'] / 100
# If 'inversion_mxn' is already normalized 0-1, dividing by 1_000_000 here might not be intended.
# Consider adjusting if you need raw investment values or a different scaling.
if 'inversion_mxn' in df.columns:
    df['inversion_mxn'] = df['inversion_mxn'] / 1_000_000

# Codificación de variables categóricas
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].astype('category').cat.codes

# 3️⃣ Agrupar metas por sector para análisis sistémico
sectores = df['sector'].unique()
print("Sectores detectados:", sectores)

# 4️⃣ Simulación Monte Carlo por sector
simulaciones = 10000
resultados_sector = {sector: [] for sector in sectores}

for _ in range(simulaciones):
    for sector in sectores:
        df_sector = df[df['sector']==sector]
        # Use 'meta_cuantitativa' as 'meta' column is not available in df_plan
        # Each indicator varies ±10% (puedes ajustar sigma por sector)
        if 'meta_cuantitativa' in df_sector.columns:
            escenario = df_sector['meta_cuantitativa'] * np.random.normal(1, 0.10, size=len(df_sector))
        else:
            print(f"Warning: 'meta_cuantitativa' not found for sector {sector}. Using zeros.")
            escenario = np.zeros(len(df_sector))
        resultados_sector[sector].append(escenario.sum())

# 5️⃣ Convertir resultados a DataFrame
mc_df = pd.DataFrame({sector: np.array(vals) for sector, vals in resultados_sector.items()})

# 6️⃣ Estadísticas por sector
stats = mc_df.describe(percentiles=[0.05, 0.5, 0.95]).T[['mean','std','5%','50%','95%']]
stats.rename(columns={'mean':'Valor Esperado','std':'Riesgo','5%':'Perc_5','50%':'Perc_50','95%':'Perc_95'}, inplace=True)
print("Estadísticas sectoriales (Valor Esperado y Riesgo):")
display(stats)

# 7️⃣ Gráfica de riesgo vs retorno por sector
plt.figure(figsize=(10,6))
plt.scatter(stats['Riesgo'], stats['Valor Esperado'], s=150, c='skyblue', edgecolors='black')
for i, sector in enumerate(stats.index):
    plt.text(stats['Riesgo'][i]*1.01, stats['Valor Esperado'][i]*1.01, sector)
plt.xlabel('Riesgo (Desviación estándar)')
plt.ylabel('Valor Esperado del cumplimiento de metas')
plt.title('Riesgo vs Retorno por Sector - Plan México 2025-2030')
plt.grid(True)
plt.show()

# 8️⃣ Distribución de resultados Monte Carlo por sector (interactiva)
fig = px.histogram(mc_df, nbins=50, title='Distribución de cumplimiento de metas por sector (Monte Carlo)')
fig.update_layout(xaxis_title='Cumplimiento total de metas', yaxis_title='Frecuencia')
fig.show()

# 9️⃣ Matriz de correlación entre sectores
correlation = mc_df.corr()
plt.figure(figsize=(10,8))
plt.matshow(correlation, fignum=1)
plt.colorbar()
plt.title('Correlación entre sectores')
plt.show()

print("Script avanzado listo: análisis sistémico, riesgo/retorno y simulación Monte Carlo sectorial.")